## Exploración de los dataframes

En este notebook se realiza la exploración de las dataframes entregados para verificar errores de estandarización.

In [1]:
import findspark
import os
from pyspark.sql.session import SparkSession
from pyspark.sql import functions as F
findspark.init()

os.chdir('..')


In [2]:
# Inicializamos spark
spark = SparkSession.builder\
    .appName('peiGo')\
    .master('local[*]')\
    .getOrCreate()

In [3]:
# Lectura de dataframes
data_catalago_comercios = spark.read.parquet('./data/raw/catalogo_comercios.parquet')
data_clientes = spark.read.parquet('./data/raw/clientes.parquet')
data_interacciones_marketing = spark.read.parquet('./data/raw/interacciones_marketing.parquet')
data_tarjetas = spark.read.parquet('./data/raw/tarjetas.parquet')
data_transacciones = spark.read.parquet('./data/raw/transacciones.parquet')

## Comercios

In [4]:
print(f'Catalogo: {data_catalago_comercios.count()}\n{data_catalago_comercios.printSchema()}')
print(data_catalago_comercios.limit(10).show())

root
 |-- comercio_codigo: string (nullable = true)
 |-- nombre_comercio: string (nullable = true)
 |-- categoria: string (nullable = true)

Catalogo: 42
None
+---------------+-------------------+------------------+
|comercio_codigo|    nombre_comercio|         categoria|
+---------------+-------------------+------------------+
|        COM-001|          Supermaxi|            retail|
|        COM-002|     Mi Comisariato|             salud|
|        COM-003|            Netflix|       restaurante|
|        COM-004|            Spotify|      supermercado|
|        COM-005|               Uber|         streaming|
|        COM-006|             Cabify|servicios_publicos|
|        COM-007|Farmacias Cruz Azul|      supermercado|
|        COM-008|           De Prati|   entretenimiento|
|        COM-009|           Cinemark|      supermercado|
|        COM-010|                CNT|         streaming|
+---------------+-------------------+------------------+

None


In [5]:
# nulos
data_catalago_comercios.select(
    [
        F.count(F.when(F.col(c).isNull() | F.isnan(c), c)).alias(c) for c in data_catalago_comercios.columns
    ]
).show()

+---------------+---------------+---------+
|comercio_codigo|nombre_comercio|categoria|
+---------------+---------------+---------+
|              0|              0|        0|
+---------------+---------------+---------+



In [6]:
data_catalago_comercios.select(F.countDistinct('nombre_comercio')).show()
data_catalago_comercios.select('nombre_comercio').distinct().orderBy('nombre_comercio').show(n=50, truncate=False)
print('hay 42 comercios, 15 nombres no genéricos')

+-------------------------------+
|count(DISTINCT nombre_comercio)|
+-------------------------------+
|                             42|
+-------------------------------+

+-------------------+
|nombre_comercio    |
+-------------------+
|CNT                |
|Cabify             |
|Cinemark           |
|Claro              |
|Comercio 16        |
|Comercio 17        |
|Comercio 18        |
|Comercio 19        |
|Comercio 20        |
|Comercio 21        |
|Comercio 22        |
|Comercio 23        |
|Comercio 24        |
|Comercio 25        |
|Comercio 26        |
|Comercio 27        |
|Comercio 28        |
|Comercio 29        |
|Comercio 30        |
|Comercio 31        |
|Comercio 32        |
|Comercio 33        |
|Comercio 34        |
|Comercio 35        |
|Comercio 36        |
|Comercio 37        |
|Comercio 38        |
|Comercio 39        |
|Comercio 40        |
|Comercio 41        |
|Comercio 42        |
|De Prati           |
|Farmacias Cruz Azul|
|KFC                |
|McDonalds     

In [7]:
data_catalago_comercios.select('categoria').distinct().orderBy('categoria').show(n=50, truncate=False)

+------------------+
|categoria         |
+------------------+
|entretenimiento   |
|restaurante       |
|retail            |
|salud             |
|servicios_publicos|
|streaming         |
|supermercado      |
|transporte        |
+------------------+



In [8]:
# verifiquemos si las categorías están bien mapeadas
data_catalago_comercios.groupBy(['categoria', 'nombre_comercio']).agg(
    F.count('*').alias('num_appears')
).orderBy(F.desc('categoria')).show(n=50, truncate=False)

# debería mapearse así
correccion_categoria_comercios = {
    'transporte':	['Uber',	'Cabify'	],
    'supermercado':	['Mi Comisariato',	'Supermaxi'	],
    'streaming':	['Spotify',	'Netflix'	],
    'telefonia':	['CNT',	'Movistar',	'Claro'], # categoria servicios_publicos se cambia a telefonia para hacer match con los comercios
    'salud':	['Farmacias Cruz Azul'		],
    'retail':	['De Prati'		],
    'restaurante':	['KFC',	'Pizza Hut',	'McDonalds'],
    'entretenimiento':	['Cinemark'		],
}

+------------------+-------------------+-----------+
|categoria         |nombre_comercio    |num_appears|
+------------------+-------------------+-----------+
|transporte        |Comercio 22        |1          |
|supermercado      |Cinemark           |1          |
|supermercado      |KFC                |1          |
|supermercado      |Comercio 37        |1          |
|supermercado      |Pizza Hut          |1          |
|supermercado      |Farmacias Cruz Azul|1          |
|supermercado      |Spotify            |1          |
|supermercado      |Comercio 29        |1          |
|streaming         |Comercio 39        |1          |
|streaming         |Comercio 42        |1          |
|streaming         |Comercio 35        |1          |
|streaming         |Comercio 24        |1          |
|streaming         |Uber               |1          |
|streaming         |CNT                |1          |
|streaming         |Comercio 31        |1          |
|streaming         |Comercio 26        |1     

## Clientes

In [9]:
data_clientes.select(
    [
        F.count(F.when(F.col(c).isNull() | F.isnan(c), c)).alias(c) for c in data_clientes.columns
    ]
).show()
print('canal_adquisicion tiene 212 nulos')

+----------+------+---------------+----------------+------+-----------------+-------------+--------------+
|cliente_id|cedula|nombre_completo|fecha_nacimiento|ciudad|canal_adquisicion|estado_cuenta|fecha_registro|
+----------+------+---------------+----------------+------+-----------------+-------------+--------------+
|         0|     0|              0|               0|     0|              212|            0|             0|
+----------+------+---------------+----------------+------+-----------------+-------------+--------------+

canal_adquisicion tiene 212 nulos


In [10]:
# exploremos los nulos
data_clientes.filter('canal_adquisicion IS NULL').show(n=15, truncate=False)

+----------+-----------+--------------------------+----------------+------------+-----------------+-------------+--------------+
|cliente_id|cedula     |nombre_completo           |fecha_nacimiento|ciudad      |canal_adquisicion|estado_cuenta|fecha_registro|
+----------+-----------+--------------------------+----------------+------------+-----------------+-------------+--------------+
|CHK-001462|1697353918 |Estefania Ortiz Andrade   |1977-05-04      |Machala     |null             |ACTIVA       |2023-03-24    |
|CHK-000902|1841332383 |Jorge Ortiz Castro        |1976-12-07      |Guayaquil   |null             |activa       |2022-07-01    |
|CHK-007908|1098870475 |Karla Chavez Mendoza      |1952-11-17      |Loja        |null             |bloqueada    |2025-11-13    |
|CHK-003685|1825149542 |Karla Zambrano Suarez     |2007-03-11      |Quito       |null             |activa       |15/10/2021    |
|CHK-006851|107560702-5|Paola Pico Perez          |02/02/1951      |Quito       |null            

In [11]:
data_clientes.select('canal_adquisicion').distinct().orderBy('canal_adquisicion').show()
print('tiene null, vacio, N/A, NULL y mayus, minus, spacios')

+--------------------+
|   canal_adquisicion|
+--------------------+
|                null|
|                    |
|        call_center |
|           organico |
| publicidad_digital |
|           referido |
|         CALL_CENTER|
|       Call_center  |
|                 N/A|
|                NULL|
|            ORGANICO|
|          Organico  |
|  PUBLICIDAD_DIGITAL|
|Publicidad_digital  |
|            REFERIDO|
|          Referido  |
|         call_center|
|            organico|
|  publicidad_digital|
|            referido|
+--------------------+

tiene null, vacio, N/A, NULL y mayus, minus, spacios


In [12]:
data_clientes.groupBy('fecha_nacimiento')\
    .agg(
        F.count('*').alias('num_appear')
    ).orderBy(F.desc('num_appear')).show()

print('campos de fechas mal escritos, estandarizar')

+----------------+----------+
|fecha_nacimiento|num_appear|
+----------------+----------+
|      1972-04-29|         5|
|      1954-11-19|         5|
|      1995-12-29|         4|
|      1976-02-21|         4|
|      1989-09-21|         4|
|      1979-08-16|         4|
|      1991-11-03|         4|
|      1998-04-10|         4|
|      1984-01-15|         4|
|      1999-06-05|         4|
|      1959-10-06|         4|
|      2004-10-16|         4|
|      2008-06-16|         4|
|      1964-10-12|         4|
|      1986-02-13|         4|
|      1999-10-11|         4|
|      2001-10-06|         4|
|      1966-08-08|         4|
|      1973-01-06|         4|
|      02/03/1996|         3|
+----------------+----------+
only showing top 20 rows

campos de fechas mal escritos, estandarizar


In [13]:
data_clientes.select(F.countDistinct('cliente_id')).show()
print('12k unicos, pero 12048 en base -> duplicados')

+--------------------------+
|count(DISTINCT cliente_id)|
+--------------------------+
|                     12000|
+--------------------------+

12k unicos, pero 12048 en base -> duplicados


In [14]:
data_clientes.groupBy('cliente_id').agg(
    F.count('*').alias('num_appear')
).orderBy(F.desc('num_appear')).show(n=50, truncate=False)

+----------+----------+
|cliente_id|num_appear|
+----------+----------+
|CHK-003398|2         |
|CHK-006444|2         |
|CHK-006477|2         |
|CHK-005970|2         |
|CHK-007164|2         |
|CHK-004274|2         |
|CHK-004804|2         |
|CHK-000018|2         |
|CHK-005960|2         |
|CHK-002489|2         |
|CHK-001989|2         |
|CHK-001166|2         |
|CHK-004641|2         |
|CHK-005186|2         |
|CHK-002031|2         |
|CHK-001174|2         |
|CHK-002419|2         |
|CHK-002117|2         |
|CHK-011788|2         |
|CHK-003506|2         |
|CHK-008549|2         |
|CHK-011159|2         |
|CHK-002228|2         |
|CHK-002662|2         |
|CHK-001903|2         |
|CHK-000136|2         |
|CHK-010283|2         |
|CHK-004465|2         |
|CHK-005934|2         |
|CHK-000947|2         |
|CHK-010897|2         |
|CHK-009630|2         |
|CHK-004595|2         |
|CHK-000036|2         |
|CHK-011223|2         |
|CHK-001294|2         |
|CHK-008102|2         |
|CHK-002712|2         |
|CHK-004637|2   

In [15]:
data_clientes.filter('cliente_id = "CHK-003398"').show(truncate=False)
print('son duplicados puros')

+----------+----------+-------------------+----------------+-------+------------------+-------------+--------------+
|cliente_id|cedula    |nombre_completo    |fecha_nacimiento|ciudad |canal_adquisicion |estado_cuenta|fecha_registro|
+----------+----------+-------------------+----------------+-------+------------------+-------------+--------------+
|CHK-003398|1184274029|Juan Pico Rodriguez|25/06/1990      | Quito |publicidad_digital|activa       |20/05/2025    |
|CHK-003398|1184274029|Juan Pico Rodriguez|25/06/1990      | Quito |publicidad_digital|activa       |20/05/2025    |
+----------+----------+-------------------+----------------+-------+------------------+-------------+--------------+

son duplicados puros


In [16]:
data_clientes.select(F.length('cedula').alias('len_cedula'))\
    .groupBy('len_cedula').agg(
        F.count('*').alias('num_personas')
    ).orderBy(F.desc('num_personas')).show(truncate=False)

data_clientes = data_clientes.withColumn(
    'len_cedula', F.length(F.col('cedula'))
)

data_clientes.filter('len_cedula > 10').show(n=15, truncate=False)

print('la cédulas están mal escritas, tiene guion')

+----------+------------+
|len_cedula|num_personas|
+----------+------------+
|10        |8435        |
|11        |3613        |
+----------+------------+

+----------+-----------+-------------------------+----------------+-----------+------------------+-------------+--------------+----------+
|cliente_id|cedula     |nombre_completo          |fecha_nacimiento|ciudad     |canal_adquisicion |estado_cuenta|fecha_registro|len_cedula|
+----------+-----------+-------------------------+----------------+-----------+------------------+-------------+--------------+----------+
|CHK-003169|174342225-8|Ana Mendoza Gonzalez     |27/07/2005      |Guayaquil  |publicidad_digital|activa       |2024-03-17    |11        |
|CHK-010631|132834933-9|David Tenesaca Salazar   |2000-11-22      |Loja       |publicidad_digital|activa       |2021-05-18    |11        |
|CHK-011560|138614198-4|Michelle Rodriguez Guaman|1974-09-03      |Guayaquil  |call_center       |activa       |18/06/2022    |11        |
|CHK-0094

In [17]:
data_clientes.select(F.countDistinct('ciudad')).show()
data_clientes.select('ciudad').distinct().show(n=50, truncate=False)
print('ciudades mal escritas, mal tipeadas')

+----------------------+
|count(DISTINCT ciudad)|
+----------------------+
|                    45|
+----------------------+

+------------+
|ciudad      |
+------------+
|Loja        |
|Portoviejo  |
|Cuenca      |
|portoviejo  |
|PORTOVIEJO  |
|RIOBAMBA    |
|GUAYAQUIL   |
|Guayaquil   |
|Quito       |
|Portoviejo  |
|Manta       |
|Manta       |
| Machala    |
|Guayaquil   |
|MANTA       |
|Loja        |
| Cuenca     |
|quito       |
|manta       |
| Portoviejo |
| Ambato     |
|AMBATO      |
|CUENCA      |
| Manta      |
|riobamba    |
|Cuenca      |
|Ambato      |
| Riobamba   |
|Machala     |
|Machala     |
|ambato      |
|MACHALA     |
|LOJA        |
|Ambato      |
|guayaquil   |
| Guayaquil  |
|machala     |
|cuenca      |
|QUITO       |
| Loja       |
|loja        |
|Riobamba    |
| Quito      |
|Riobamba    |
|Quito       |
+------------+

ciudades mal escritas, mal tipeadas


In [18]:
data_clientes.select(F.countDistinct('estado_cuenta')).show()
data_clientes.select('estado_cuenta').distinct().show(n=50, truncate=False)
print('estado_cuenta: mal estandarizado, mayus, minus, spaces')

+-----------------------------+
|count(DISTINCT estado_cuenta)|
+-----------------------------+
|                           16|
+-----------------------------+

+-------------+
|estado_cuenta|
+-------------+
|INACTIVA     |
| inactiva    |
|ACTIVA       |
|cerrada      |
|Activa       |
|Cerrada      |
| cerrada     |
|CERRADA      |
|Inactiva     |
|Bloqueada    |
| bloqueada   |
| activa      |
|BLOQUEADA    |
|inactiva     |
|bloqueada    |
|activa       |
+-------------+

estado_cuenta: mal estandarizado, mayus, minus, spaces


## Interacciones

In [19]:
data_interacciones_marketing.select(
    [
        F.count(F.when(F.col(c).isNull() | F.isnan(c), c)).alias(c) for c in data_interacciones_marketing.columns
    ]
).show()
print('Hay nulos en la columna respondio')

+--------------+----------+-------+--------------+-----+---------+
|interaccion_id|cliente_id|campana|fecha_contacto|canal|respondio|
+--------------+----------+-------+--------------+-----+---------+
|             0|         0|      0|             0|    0|     2004|
+--------------+----------+-------+--------------+-----+---------+

Hay nulos en la columna respondio


In [20]:
data_interacciones_marketing.select(F.countDistinct('cliente_id')).show()
data_interacciones_marketing.filter('cliente_id = "CHK-004008"').show()
print('11769 clientes únicos en transaccion, el resto no ha hecho nada')

+--------------------------+
|count(DISTINCT cliente_id)|
+--------------------------+
|                     11769|
+--------------------------+

+--------------+----------+--------------------+--------------+--------+---------+
|interaccion_id|cliente_id|             campana|fecha_contacto|   canal|respondio|
+--------------+----------+--------------------+--------------+--------+---------+
|   MKT-0000001|CHK-004008|     Cashback Verano|    2023-07-18|   email|        0|
|   MKT-0004019|CHK-004008|bienvenida nuevos...|    2026-05-15|WHATSAPP|    false|
|   MKT-0004647|CHK-004008|Reactivacion Ahorros|    2023-06-09|    push|     true|
|   MKT-0016298|CHK-004008|     Cashback Verano|    2021-10-19|    push|    False|
|   MKT-0023204|CHK-004008|Bienvenida Nuevos...|    2021-12-02|    push|    False|
|   MKT-0026841|CHK-004008|Piloto Tarjeta Fi...|    2024-05-26|     sms|        1|
+--------------+----------+--------------------+--------------+--------+---------+

11769 clientes únicos e

In [21]:
data_interacciones_marketing.select(F.countDistinct('campana')).show()
data_interacciones_marketing.select('campana').distinct().show(n=50, truncate=False)

print('campana: mal estandarizado, espacios, sin guion bajo, mayus, minus')

+-----------------------+
|count(DISTINCT campana)|
+-----------------------+
|                     15|
+-----------------------+

+--------------------------+
|campana                   |
+--------------------------+
|REACTIVACION AHORROS      |
|REFERIDOS Q3              |
|CASHBACK VERANO           |
|Bienvenida Nuevos Usuarios|
|PILOTO TARJETA FISICA Q2  |
|BIENVENIDA NUEVOS USUARIOS|
|reactivacion ahorros      |
|Reactivacion Ahorros      |
|piloto tarjeta fisica q2  |
|cashback verano           |
|referidos q3              |
|Cashback Verano           |
|Piloto Tarjeta Fisica Q2  |
|bienvenida nuevos usuarios|
|Referidos Q3              |
+--------------------------+

campana: mal estandarizado, espacios, sin guion bajo, mayus, minus


In [22]:
data_interacciones_marketing.select(F.countDistinct('fecha_contacto')).show()
data_interacciones_marketing.select('fecha_contacto').distinct().show(n=50, truncate=False)

print('fecha_contacto: hay fechas de todos los formatos, 20/09/2024, 2023-01-21, 1694044800000 y espacios')

+------------------------------+
|count(DISTINCT fecha_contacto)|
+------------------------------+
|                          5178|
+------------------------------+

+--------------+
|fecha_contacto|
+--------------+
|2023-01-21    |
|2024-07-14    |
|2024-10-24    |
|2024-09-15    |
|2023-05-01    |
|2024-01-19    |
|25/05/2021    |
|2021-11-03    |
|2023-05-18    |
|2024-08-20    |
|22/06/2022    |
|2022-10-05    |
|18/12/2022    |
|2026-02-01    |
|30/07/2024    |
|16/12/2025    |
|01/11/2024    |
|11/03/2026    |
|1694044800000 |
|1616544000000 |
|08/01/2025    |
|20/09/2021    |
|1661990400000 |
|02/04/2024    |
|1686700800000 |
|1659139200000 |
|30/05/2025    |
|13/04/2025    |
|2023-04-28    |
|2025-02-25    |
|2021-12-23    |
|2023-04-21    |
|24/06/2025    |
|2023-04-17    |
|2025-06-03    |
|09/04/2026    |
|2022-10-07    |
|21/02/2026    |
|2024-08-06    |
|2024-10-22    |
|1660694400000 |
|13/06/2023    |
|15/12/2023    |
|12/01/2023    |
|30/08/2025    |
|1626998400000 |
|

In [23]:
data_interacciones_marketing.select(F.countDistinct('canal')).show()
data_interacciones_marketing.select('canal').distinct().show(n=50, truncate=False)

print('canal: mayus, minus')

+---------------------+
|count(DISTINCT canal)|
+---------------------+
|                    8|
+---------------------+

+--------+
|canal   |
+--------+
|WHATSAPP|
|SMS     |
|whatsapp|
|email   |
|EMAIL   |
|sms     |
|push    |
|PUSH    |
+--------+

canal: mayus, minus


In [24]:
data_interacciones_marketing.select(F.countDistinct('respondio')).show()
data_interacciones_marketing\
    .groupBy('respondio').agg(
        F.count('*').alias('num_appear')
    ).orderBy(F.desc('num_appear')).show()

data_interacciones_marketing.filter('respondio = "No"').show(truncate=False)

print('respondio: debe ser bool, tiene False, false, No, True, true, 0, 1, nan')

+-------------------------+
|count(DISTINCT respondio)|
+-------------------------+
|                        9|
+-------------------------+

+---------+----------+
|respondio|num_appear|
+---------+----------+
|    False|     16808|
|     True|      4684|
|    false|      2155|
|       No|      2096|
|        1|      2082|
|        0|      2071|
|     true|      2055|
|       Sí|      2045|
|      nan|      2004|
+---------+----------+

+--------------+----------+--------------------------+--------------+--------+---------+
|interaccion_id|cliente_id|campana                   |fecha_contacto|canal   |respondio|
+--------------+----------+--------------------------+--------------+--------+---------+
|MKT-0000018   |CHK-003046|bienvenida nuevos usuarios|2026-04-07    |push    |No       |
|MKT-0000021   |CHK-000014|PILOTO TARJETA FISICA Q2  |2025-03-02    |email   |No       |
|MKT-0000024   |CHK-008772|Reactivacion Ahorros      |2021-02-08    |WHATSAPP|No       |
|MKT-0000026   |CHK-00541

## Tarjetas

In [25]:
data_tarjetas.select(
    [
        F.count(F.when(F.col(c).isNull() | F.isnan(c), c)).alias(c) for c in data_tarjetas.columns
    ]
).show()
print('hay nulos en fecha_emision y fecha activación')

+----------+----------+----+-------------+-------------+----------------+
|tarjeta_id|cliente_id|tipo|estado_codigo|fecha_emision|fecha_activacion|
+----------+----------+----+-------------+-------------+----------------+
|         0|         0|   0|            0|            8|            2817|
+----------+----------+----+-------------+-------------+----------------+

hay nulos en fecha_emision y fecha activación


In [26]:
data_tarjetas.select(F.countDistinct('tarjeta_id')).show()

data_tarjetas.groupBy(
    'tarjeta_id'
).agg(
    F.count('*').alias('num_appear')
).orderBy(F.desc('num_appear')).show()

data_tarjetas.filter('tarjeta_id = "TRJ-012592"').show(truncate=False)

print('tarjeta_id: hay duplicados, hay 16585 id únicos, pero 16634 en base')

+--------------------------+
|count(DISTINCT tarjeta_id)|
+--------------------------+
|                     16585|
+--------------------------+

+----------+----------+
|tarjeta_id|num_appear|
+----------+----------+
|TRJ-001028|         2|
|TRJ-001629|         2|
|TRJ-002274|         2|
|TRJ-003115|         2|
|TRJ-000076|         2|
|TRJ-009077|         2|
|TRJ-000159|         2|
|TRJ-016188|         2|
|TRJ-003255|         2|
|TRJ-012099|         2|
|TRJ-006524|         2|
|TRJ-013457|         2|
|TRJ-009464|         2|
|TRJ-016101|         2|
|TRJ-010695|         2|
|TRJ-012592|         2|
|TRJ-011129|         2|
|TRJ-000516|         2|
|TRJ-007765|         2|
|TRJ-007749|         2|
+----------+----------+
only showing top 20 rows

+----------+----------+-------+-------------+-------------+----------------+
|tarjeta_id|cliente_id|tipo   |estado_codigo|fecha_emision|fecha_activacion|
+----------+----------+-------+-------------+-------------+----------------+
|TRJ-012592|CHK-00910

In [27]:
data_tarjetas.select(F.countDistinct('cliente_id')).show()
data_tarjetas.groupBy(
    'cliente_id'
).agg(
    F.count('*').alias('num_appear')
).orderBy(F.desc('num_appear')).show()

data_tarjetas.filter('cliente_id == "CHK-002699"').show() # duplicado

print('cliente_id: 11691 de los 12mil tienen tarjeta')

+--------------------------+
|count(DISTINCT cliente_id)|
+--------------------------+
|                     11691|
+--------------------------+

+----------+----------+
|cliente_id|num_appear|
+----------+----------+
|CHK-002699|         3|
|CHK-004027|         3|
|CHK-000112|         3|
|CHK-008769|         3|
|CHK-001808|         3|
|CHK-011634|         3|
|CHK-008078|         3|
|CHK-000380|         3|
|CHK-011345|         3|
|CHK-009108|         3|
|CHK-009738|         3|
|CHK-011314|         3|
|CHK-001636|         3|
|CHK-006068|         3|
|CHK-006577|         3|
|CHK-005814|         3|
|CHK-004130|         3|
|CHK-008917|         3|
|CHK-002257|         3|
|CHK-001162|         3|
+----------+----------+
only showing top 20 rows

+----------+----------+-------+-------------+-------------+----------------+
|tarjeta_id|cliente_id|   tipo|estado_codigo|fecha_emision|fecha_activacion|
+----------+----------+-------+-------------+-------------+----------------+
|TRJ-003741|CHK-00269

In [28]:
data_tarjetas.select(F.countDistinct('tipo')).show()
data_tarjetas.groupBy(
    'tipo'
).agg(
    F.count('*').alias('num_appear')
).orderBy(F.desc('num_appear')).show(n=50, truncate=False)

print('tipo: estandarizar virtual, fiscia, física, ...')

+--------------------+
|count(DISTINCT tipo)|
+--------------------+
|                  11|
+--------------------+

+--------+----------+
|tipo    |num_appear|
+--------+----------+
|virtual |9213      |
|fisica  |4428      |
|VIRTUAL |536       |
|Virtual |505       |
|virtual |500       |
|V       |485       |
| fisica |217       |
|F       |192       |
|Fisica  |190       |
|FISICA  |187       |
|física  |181       |
+--------+----------+

tipo: estandarizar virtual, fiscia, física, ...


In [29]:
data_tarjetas.filter('cliente_id == "CHK-004008"').show()

print('fecha_activacion: todos los formatos y nulos')
print('fecha_emision: todos los formatos')

+----------+----------+-------+-------------+-------------+----------------+
|tarjeta_id|cliente_id|   tipo|estado_codigo|fecha_emision|fecha_activacion|
+----------+----------+-------+-------------+-------------+----------------+
|TRJ-005518|CHK-004008| fisica|            1|   2026-05-01|      25/05/2026|
|TRJ-005517|CHK-004008|virtual|            1|   2021-05-11|      18/05/2021|
+----------+----------+-------+-------------+-------------+----------------+

fecha_activacion: todos los formatos y nulos
fecha_emision: todos los formatos


In [30]:
data_tarjetas.groupBy(
    'estado_codigo'
).agg(
    F.count('*').alias('num_appear')
).orderBy(F.desc('num_appear')).show(n=50, truncate=False)


+-------------+----------+
|estado_codigo|num_appear|
+-------------+----------+
|1            |14025     |
|2            |1748      |
|3            |861       |
+-------------+----------+



In [31]:
data_tarjetas.filter('estado_codigo = 1').show()

+----------+----------+-------+-------------+-------------+----------------+
|tarjeta_id|cliente_id|   tipo|estado_codigo|fecha_emision|fecha_activacion|
+----------+----------+-------+-------------+-------------+----------------+
|TRJ-004448|CHK-003226| fisica|            1|   2026-07-01|            null|
|TRJ-002909|CHK-002104|virtual|            1|   2022-09-16|      2022-09-19|
|TRJ-012662|CHK-009159|virtual|            1|   2026-01-13|      2026-01-18|
|TRJ-006817|CHK-004955|virtual|            1|   2023-12-03|      2023-12-06|
|TRJ-010895|CHK-007913|virtual|            1|   2021-01-31|      05/02/2021|
|TRJ-001774|CHK-001273|virtual|            1|   2023-08-23|      2023-08-27|
|TRJ-008679|CHK-006301|virtual|            1|   2024-03-03|      2024-03-08|
|TRJ-013514|CHK-009779|virtual|            1|   30/12/2021|      2021-12-30|
|TRJ-014653|CHK-010597|virtual|            1|   2022-12-15|      2022-12-12|
|TRJ-005955|CHK-004324| fisica|            1|   2026-05-01|            null|

## Transacciones

In [32]:
data_transacciones.select(
    [
        F.count(F.when(F.col(c).isNull() | F.isnan(c), c)).alias(c) for c in data_transacciones.columns
    ]
).show()

print('hay nulos en comercio_codigo')

+--------------+----------+-----+----------------+-----+---------------+-------------+
|transaccion_id|cliente_id|fecha|tipo_transaccion|monto|comercio_codigo|es_devolucion|
+--------------+----------+-----+----------------+-----+---------------+-------------+
|             0|         0|    0|               0|    0|         175040|            0|
+--------------+----------+-----+----------------+-----+---------------+-------------+

hay nulos en comercio_codigo


In [33]:
data_transacciones.select(F.countDistinct('transaccion_id')).show()
data_transacciones.groupby(
    'transaccion_id'
).agg(
    F.count('*').alias('num_appear')
).orderBy(F.desc('num_appear')).show()

data_transacciones.filter('transaccion_id == "TX-00179393"').show()

print('transaccion_id: duplicados')

+------------------------------+
|count(DISTINCT transaccion_id)|
+------------------------------+
|                        193015|
+------------------------------+

+--------------+----------+
|transaccion_id|num_appear|
+--------------+----------+
|   TX-00179393|         2|
|   TX-00167641|         2|
|   TX-00039127|         2|
|   TX-00105778|         2|
|   TX-00157381|         2|
|   TX-00084593|         2|
|   TX-00084087|         2|
|   TX-00023126|         2|
|   TX-00024236|         2|
|   TX-00015832|         2|
|   TX-00048397|         2|
|   TX-00020292|         2|
|   TX-00049262|         2|
|   TX-00146943|         2|
|   TX-00164880|         2|
|   TX-00054904|         2|
|   TX-00055128|         2|
|   TX-00144128|         2|
|   TX-00183302|         2|
|   TX-00167860|         2|
+--------------+----------+
only showing top 20 rows

+--------------+----------+----------+----------------+-----+---------------+-------------+
|transaccion_id|cliente_id|     fecha|tipo_t

In [34]:
data_transacciones.select('tipo_transaccion').distinct().orderBy('tipo_transaccion').show(n=50, truncate=False)
data_transacciones.groupby(
    'tipo_transaccion'
).agg(
    F.count('*').alias('num_appear')
).orderBy(F.desc('num_appear')).show(n=50, truncate=False)

print('tipo_transaccion: mal estandarizado, espacios, sin guines bajos, mayus, mminus')

+----------------+
|tipo_transaccion|
+----------------+
| cash_in        |
| cash_out       |
| compra_tarjeta |
| p2p_in         |
| p2p_out        |
| pago_servicio  |
| recarga_celular|
| remesa         |
|CASH_IN         |
|CASH_OUT        |
|COMPRA_TARJETA  |
|Cash In         |
|Cash Out        |
|Compra Tarjeta  |
|P2P In          |
|P2P Out         |
|P2P_IN          |
|P2P_OUT         |
|PAGO_SERVICIO   |
|Pago Servicio   |
|RECARGA_CELULAR |
|REMESA          |
|Recarga Celular |
|Remesa          |
|cash_in         |
|cash_out        |
|compra_tarjeta  |
|p2p_in          |
|p2p_out         |
|pago_servicio   |
|recarga_celular |
|remesa          |
+----------------+

+----------------+----------+
|tipo_transaccion|num_appear|
+----------------+----------+
|cash_in         |41879     |
|cash_out        |28091     |
|p2p_in          |21073     |
|p2p_out         |20923     |
|compra_tarjeta  |15195     |
|recarga_celular |14087     |
|pago_servicio   |7079      |
|remesa        

In [35]:
data_transacciones.filter('monto = "nan"').show()

print('monto: verificar si hay str')

+--------------+----------+-----+----------------+-----+---------------+-------------+
|transaccion_id|cliente_id|fecha|tipo_transaccion|monto|comercio_codigo|es_devolucion|
+--------------+----------+-----+----------------+-----+---------------+-------------+
+--------------+----------+-----+----------------+-----+---------------+-------------+

monto: verificar si hay str


In [36]:
data_transacciones.select(F.countDistinct('comercio_codigo')).show()

data_transacciones.groupby(
    'comercio_codigo'
).agg(
    F.count('*').alias('num_appear')
).orderBy(F.desc('num_appear')).show(n=50, truncate=False)
print('solo están los 16 comercios no genéricos')

data_transacciones.filter('comercio_codigo IS NULL').select('tipo_transaccion').distinct().orderBy('tipo_transaccion').show(n=50, truncate=False)
data_transacciones.filter('comercio_codigo IS NOT NULL').select('tipo_transaccion').distinct().orderBy('tipo_transaccion').show(n=50, truncate=False)

print('''
comercio_codigo: 194173 transacciones, 175040 nulos
al parecer solo los tipo_transaccion = compra_tarjeta tienen comercio_codigo
igual, se debe estandarizar 
''')


+-------------------------------+
|count(DISTINCT comercio_codigo)|
+-------------------------------+
|                             44|
+-------------------------------+

+---------------+----------+
|comercio_codigo|num_appear|
+---------------+----------+
|null           |175040    |
|COM-037        |486       |
|COM-011        |479       |
|COM-010        |470       |
|COM-015        |465       |
|COM-023        |465       |
|COM-035        |462       |
|COM-016        |460       |
|COM-007        |459       |
|COM-027        |456       |
|COM-030        |456       |
|COM-029        |454       |
|COM-036        |452       |
|COM-012        |445       |
|COM-041        |443       |
|COM-021        |443       |
|COM-006        |439       |
|COM-025        |439       |
|COM-998        |437       |
|COM-001        |433       |
|COM-014        |433       |
|COM-009        |432       |
|COM-017        |432       |
|COM-033        |431       |
|COM-999        |429       |
|COM-018        |

In [37]:
data_transacciones.select('es_devolucion').distinct().orderBy('es_devolucion').show(n=50, truncate=False)

print('es_devolucion: tiene Mezcla de todo, debería ser booleano')

+-------------+
|es_devolucion|
+-------------+
|0            |
|1            |
|FALSE        |
|False        |
|N            |
|No           |
|S            |
|Si           |
|TRUE         |
|True         |
+-------------+

es_devolucion: tiene Mezcla de todo, debería ser booleano


In [38]:
#* análisis de un caso específico
cliente = 'CHK-004008'
cliente = 'CHK-010631'
data_transacciones.filter(f'cliente_id == "{cliente}"').show(truncate=False, n=50)
data_tarjetas.filter(f'cliente_id == "{cliente}"').show(truncate=False, n=50)
data_interacciones_marketing.filter(f'cliente_id == "{cliente}"').show(truncate=False, n=50)
data_clientes.filter(f'cliente_id == "{cliente}"').show(truncate=False, n=50)

data_transacciones.filter('tipo_transaccion == "compra_tarjeta"').select(F.min('fecha'), F.max('fecha')).show()

+--------------+----------+----------+----------------+------+---------------+-------------+
|transaccion_id|cliente_id|fecha     |tipo_transaccion|monto |comercio_codigo|es_devolucion|
+--------------+----------+----------+----------------+------+---------------+-------------+
|TX-00170912   |CHK-010631|26/10/2024|p2p_in          |10.05 |null           |False        |
|TX-00170917   |CHK-010631|24/09/2025|cash_out        |26.74 |null           |0            |
|TX-00170913   |CHK-010631|2025-12-27|recarga_celular |36.77 |null           |False        |
|TX-00170909   |CHK-010631|2022-11-08|cash_in         |$7.53 |null           |False        |
|TX-00170908   |CHK-010631|2025-07-31| recarga_celular|38.95 |null           |No           |
|TX-00170907   |CHK-010631|2025-04-21|cash_out        |25.42 |null           |False        |
|TX-00170911   |CHK-010631|2022-03-08|p2p_in          |12.36 |null           |S            |
|TX-00170918   |CHK-010631|07/12/2023|pago_servicio   |22.85 |null    